# Rossmann Store Sales — Data Cleaning

## Goal
Prepare the raw sales and store files for analysis and modeling.

The raw CSV files are kept unchanged.


## 1. Setup
Import libraries and locate the project data folders.


In [ ]:
from pathlib import Path

import pandas as pd

In [2]:
candidate_roots = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "data" / "raw").exists()),
    Path.cwd().parent,
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = RAW_DATA_DIR / "train.csv"
STORE_PATH = RAW_DATA_DIR / "store.csv"

assert TRAIN_PATH.exists(), f"File not found: {TRAIN_PATH}"
assert STORE_PATH.exists(), f"File not found: {STORE_PATH}"

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/ebrietas/rossmann-sales-forecasting


## 2. Load Raw Data
Load the original CSV files and create working copies.


In [3]:
train = pd.read_csv(TRAIN_PATH, low_memory=False)
store = pd.read_csv(STORE_PATH)

train_clean = train.copy()
store_clean = store.copy()

print(f"Training data shape: {train.shape}")
print(f"Store metadata shape: {store.shape}")

Training data shape: (1017209, 9)
Store metadata shape: (1115, 10)


## 3. Standardize `StateHoliday`
Convert `StateHoliday` to a consistent string category.


In [4]:
train_clean["StateHoliday"] = train_clean["StateHoliday"].astype("string")

display(
    train_clean["StateHoliday"]
    .value_counts(dropna=False)
    .rename_axis("StateHoliday")
    .reset_index(name="Rows")
)

print("Data type:", train_clean["StateHoliday"].dtype)
print("Unique categories:", train_clean["StateHoliday"].nunique(dropna=False))

,StateHoliday,Rows
0,0,986159
1,a,20260
2,b,6690
3,c,4100


Data type: string
Unique categories: 4


The column now has four categories: `0`, `a`, `b`, and `c`.


## 4. Convert `Date`
Convert `Date` to Pandas datetime.


In [5]:
train_clean["Date"] = pd.to_datetime(
    train_clean["Date"],
    format="%Y-%m-%d",
    errors="raise",
)

date_validation = pd.Series(
    {
        "Data type": str(train_clean["Date"].dtype),
        "Missing dates": train_clean["Date"].isna().sum(),
        "First date": train_clean["Date"].min(),
        "Last date": train_clean["Date"].max(),
    }
)

display(date_validation.to_frame("Value"))

,Value
Data type,datetime64[us]
Missing dates,0
First date,2013-01-01 00:00:00
Last date,2015-07-31 00:00:00


The date range is unchanged: 2013-01-01 to 2015-07-31. Invalid dates would raise an error.


## 5. Convert Discrete Metadata
Check the Promo2 and competition month, week, and year columns before converting them to nullable integers.


In [6]:
integer_like_columns = [
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear",
]

integer_validation = {}

for column in integer_like_columns:
    non_missing = store_clean[column].dropna()
    integer_validation[column] = int(((non_missing % 1) != 0).sum())

display(pd.Series(integer_validation, name="Non-integer observed values").to_frame())

,Non-integer observed values
CompetitionOpenSinceMonth,0
CompetitionOpenSinceYear,0
Promo2SinceWeek,0
Promo2SinceYear,0


In [7]:
for column in integer_like_columns:
    store_clean[column] = store_clean[column].astype("Int64")

display(store_clean[integer_like_columns].dtypes.to_frame("Data_Type"))
display(store_clean[integer_like_columns].isna().sum().to_frame("Missing_Values"))

,Data_Type
CompetitionOpenSinceMonth,Int64
CompetitionOpenSinceYear,Int64
Promo2SinceWeek,Int64
Promo2SinceYear,Int64


,Missing_Values
CompetitionOpenSinceMonth,354
CompetitionOpenSinceYear,354
Promo2SinceWeek,544
Promo2SinceYear,544


The observed values are integer-like. Nullable `Int64` keeps missing values without using decimal types.


## 6. Review Missing Values
Check why Promo2 and competition metadata are missing before deciding what to do.


### Promo2 metadata
For stores without Promo2, the Promo2 start fields do not apply.


In [8]:
promo2_metadata_columns = [
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
]

promo2_inactive = store_clean["Promo2"].eq(0)
promo2_active = store_clean["Promo2"].eq(1)

promo2_validation = pd.DataFrame(
    {
        "Stores": [
            int(promo2_inactive.sum()),
            int(promo2_active.sum()),
        ],
        "Rows with all metadata missing": [
            int(
                store_clean.loc[
                    promo2_inactive,
                    promo2_metadata_columns,
                ]
                .isna()
                .all(axis=1)
                .sum()
            ),
            int(
                store_clean.loc[
                    promo2_active,
                    promo2_metadata_columns,
                ]
                .isna()
                .all(axis=1)
                .sum()
            ),
        ],
        "Missing metadata cells": [
            int(
                store_clean.loc[
                    promo2_inactive,
                    promo2_metadata_columns,
                ]
                .isna()
                .sum()
                .sum()
            ),
            int(
                store_clean.loc[
                    promo2_active,
                    promo2_metadata_columns,
                ]
                .isna()
                .sum()
                .sum()
            ),
        ],
    },
    index=["Promo2 = 0", "Promo2 = 1"],
)

display(promo2_validation)

,Stores,Rows with all metadata missing,Missing metadata cells
Promo2 = 0,544,544,1632
Promo2 = 1,571,0,0


Keep these missing values. They are expected for stores with `Promo2 = 0`; no imputation is needed.


### Competition metadata
Check the missing-value patterns for competition distance and opening dates.


In [9]:
competition_columns = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
]

competition_missingness = (
    store_clean[competition_columns]
    .isna()
    .value_counts()
    .rename("Store_Count")
    .reset_index()
)

display(competition_missingness)

,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Store_Count
0,False,False,False,761
1,False,True,True,351
2,True,True,True,3


The patterns are:
- 761 stores have all competition fields.
- 351 stores have distance but no opening date.
- 3 stores have all competition fields missing.

Keep these values missing because the true information is unknown.


### Missing competition distance
Inspect the stores with no recorded competition distance.


In [10]:
display(
    store_clean.loc[
        store_clean["CompetitionDistance"].isna(),
        [
            "Store",
            "CompetitionDistance",
            "CompetitionOpenSinceMonth",
            "CompetitionOpenSinceYear",
            "StoreType",
            "Assortment",
            "Promo2",
        ],
    ]
)

,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,StoreType,Assortment,Promo2
290,291,NaN,<NA>,<NA>,d,a,0
621,622,NaN,<NA>,<NA>,a,c,0
878,879,NaN,<NA>,<NA>,d,a,1


The source data does not provide the missing distances, so they remain missing.


## 7. Standardize Store Categories
Convert the store category columns to string types.


In [11]:
categorical_store_columns = [
    "StoreType",
    "Assortment",
    "PromoInterval",
]

for column in categorical_store_columns:
    store_clean[column] = store_clean[column].astype("string")

display(store_clean[categorical_store_columns].dtypes.to_frame("Data_Type"))
display(store_clean[categorical_store_columns].isna().sum().to_frame("Missing_Values"))

,Data_Type
StoreType,string
Assortment,string
PromoInterval,string


,Missing_Values
StoreType,0
Assortment,0
PromoInterval,544


`StoreType` and `Assortment` have no missing values. Missing `PromoInterval` values belong to stores without Promo2.


## 8. Validate Cleaned Data
Check row counts, data types, missing values, and duplicate store-date pairs.


In [12]:
final_checks = pd.DataFrame(
    {
        "Check": [
            "Training row count preserved",
            "Store metadata row count preserved",
            "Duplicate training rows",
            "Duplicate store metadata rows",
            "Duplicate Store-Date combinations",
            "Training stores missing from store metadata",
            "Store metadata IDs missing from training",
        ],
        "Result": [
            len(train_clean) == len(train),
            len(store_clean) == len(store),
            int(train_clean.duplicated().sum()),
            int(store_clean.duplicated().sum()),
            int(train_clean.duplicated(["Store", "Date"]).sum()),
            len(set(train_clean["Store"]) - set(store_clean["Store"])),
            len(set(store_clean["Store"]) - set(train_clean["Store"])),
        ],
    }
)

display(final_checks)

,Check,Result
0,Training row count preserved,True
1,Store metadata row count preserved,True
2,Duplicate training rows,0
3,Duplicate store metadata rows,0
4,Duplicate Store-Date combinations,0
5,Training stores missing from store metadata,0
6,Store metadata IDs missing from training,0


In [13]:
print("Training missing values")
display(train_clean.isna().sum().to_frame("Missing_Values"))

print("Store metadata missing values")
display(store_clean.isna().sum().to_frame("Missing_Values"))

print("Training data types")
display(train_clean.dtypes.to_frame("Data_Type"))

print("Store metadata data types")
display(store_clean.dtypes.to_frame("Data_Type"))

Training missing values


,Missing_Values
Store,0
DayOfWeek,0
Date,0
Sales,0
Customers,0
Open,0
Promo,0
StateHoliday,0
SchoolHoliday,0


Store metadata missing values


,Missing_Values
Store,0
StoreType,0
Assortment,0
CompetitionDistance,3
CompetitionOpenSinceMonth,354
CompetitionOpenSinceYear,354
Promo2,0
Promo2SinceWeek,544
Promo2SinceYear,544
PromoInterval,544


Training data types


,Data_Type
Store,int64
DayOfWeek,int64
Date,datetime64[us]
Sales,int64
Customers,int64
Open,int64
Promo,int64
StateHoliday,string
SchoolHoliday,int64


Store metadata data types


,Data_Type
Store,int64
StoreType,string
Assortment,string
CompetitionDistance,float64
CompetitionOpenSinceMonth,Int64
CompetitionOpenSinceYear,Int64
Promo2,int64
Promo2SinceWeek,Int64
Promo2SinceYear,Int64
PromoInterval,string


### Cleaning Summary

The following cleaning steps were completed:

- No rows were removed from either dataset.
- `train_clean` contains no missing values.
- `StateHoliday` was standardized into four string categories: `0`, `a`, `b`, and `c`.
- `Date` was converted to a Pandas datetime type.
- Discrete metadata columns were converted to nullable `Int64` types to preserve missing values.
- Store category columns were standardized as strings.

Missing values were intentionally preserved:

- The missing Promo2 metadata for 544 stores with `Promo2 = 0` is expected because Promo2 does not apply to these stores.
- 351 stores have a competition distance but no recorded competition opening date.
- 3 stores have all three competition fields missing.

No unsupported assumptions were made to fill unknown competition information.

The original row counts and Store-Date uniqueness were preserved throughout the cleaning process.

## 9. Save Cleaned Data
Save the cleaned tables as Parquet files.


In [14]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CLEAN_PATH = PROCESSED_DATA_DIR / "train_clean.parquet"
STORE_CLEAN_PATH = PROCESSED_DATA_DIR / "store_clean.parquet"

train_clean.to_parquet(
    TRAIN_CLEAN_PATH,
    index=False,
)

store_clean.to_parquet(
    STORE_CLEAN_PATH,
    index=False,
)

print(f"Saved: {TRAIN_CLEAN_PATH}")
print(f"Saved: {STORE_CLEAN_PATH}")

Saved: /Users/ebrietas/rossmann-sales-forecasting/data/processed/train_clean.parquet
Saved: /Users/ebrietas/rossmann-sales-forecasting/data/processed/store_clean.parquet


## 10. Verify Saved Files
Reload the saved files and check that they can be read.


In [15]:
train_processed = pd.read_parquet(TRAIN_CLEAN_PATH)
store_processed = pd.read_parquet(STORE_CLEAN_PATH)

round_trip_validation = pd.DataFrame(
    {
        "Check": [
            "Training shape preserved",
            "Store metadata shape preserved",
            "Training dtypes preserved",
            "Store metadata dtypes preserved",
            "Training values preserved",
            "Store metadata values preserved",
        ],
        "Passed": [
            train_processed.shape == train_clean.shape,
            store_processed.shape == store_clean.shape,
            train_processed.dtypes.equals(train_clean.dtypes),
            store_processed.dtypes.equals(store_clean.dtypes),
            train_processed.equals(train_clean),
            store_processed.equals(store_clean),
        ],
    }
)

display(round_trip_validation)

,Check,Passed
0,Training shape preserved,True
1,Store metadata shape preserved,True
2,Training dtypes preserved,True
3,Store metadata dtypes preserved,True
4,Training values preserved,True
5,Store metadata values preserved,True


### Outcome

All round-trip validation checks passed after reloading the saved Parquet files.

The following properties were preserved:

- Training dataset shape
- Store metadata shape
- Training dataset data types
- Store metadata data types
- Training dataset values
- Store metadata values

The cleaned datasets are now ready for exploratory data analysis and machine learning.

The original CSV files remain unchanged.

Missing Promo2 and competition metadata were intentionally preserved because they represent information that is either not applicable or unavailable in the source data.